In [ ]:
import pandas as pd
import re

### 1. ClinVar 로드 및 protein_variant 추출
clin = pd.read_csv(r"E:\CAGI_data\variant_summary.txt", sep="\t", low_memory=False)

# missense 변이 필터
clin = clin[
    (clin["Type"] == "single nucleotide variant") &
    (clin["Assembly"] == "GRCh38") &
    (clin["Name"].str.contains(r"\(p\.", na=False))
]

# 예: p.Gly1046Arg → G1046R
def name_to_variant(name_field):
    match = re.search(r"\(p\.([A-Z][a-z]{2})(\d+)([A-Z][a-z]{2})\)", name_field)
    aa3_to1 = {
        'Ala':'A','Cys':'C','Asp':'D','Glu':'E','Phe':'F','Gly':'G','His':'H',
        'Ile':'I','Lys':'K','Leu':'L','Met':'M','Asn':'N','Pro':'P','Gln':'Q',
        'Arg':'R','Ser':'S','Thr':'T','Val':'V','Trp':'W','Tyr':'Y','Ter':'*'
    }
    if not match:
        return None
    wt = aa3_to1.get(match[1], 'X')
    pos = match[2]
    mt = aa3_to1.get(match[3], 'X')
    return f"{wt}{pos}{mt}"

clin["protein_variant"] = clin["Name"].apply(name_to_variant)
clin = clin.dropna(subset=["protein_variant"])
clin = clin[clin["protein_variant"].str.match(r"^[A-Z]\d+[A-Z]$")]

### 2. 라벨 부여 (ClinVar 기준)
def get_label(clinsig):
    clinsig = str(clinsig).lower()
    if "conflict" in clinsig:
        return None  # 제외
    if "pathogenic" in clinsig:
        return 1     # likely pathogenic 포함
    elif "benign" in clinsig:
        return 0     # likely benign 포함
    else:
        return None  # VUS, unknown, etc. → 제외

clin["Label"] = clin["ClinicalSignificance"].apply(get_label)
clin = clin[clin["Label"].notnull()]
clin["Label"] = clin["Label"].astype(int)


In [31]:
clin["ReferenceAlleleVCF"].unique()

array(['A', 'G', 'T', 'C'], dtype=object)

In [33]:
clin["AlternateAlleleVCF"].unique()

array(['G', 'A', 'T', 'C', 'Y', 'R', 'N'], dtype=object)

In [37]:
# UniProtKB 추출 함수
def extract_uniprot_id(otherids):
    if pd.isna(otherids):
        return None
    match = re.search(r"UniProtKB:(\w+)", otherids)
    return match.group(1) if match else None

# UniProtID 추가
clin["UniProtID"] = clin["OtherIDs"].apply(extract_uniprot_id)

In [38]:
# UniProtKB 추출 함수
def extract_uniprot_id(otherids):
    if pd.isna(otherids):
        return None
    match = re.search(r"UniProtKB:(\w+)", otherids)
    return match.group(1) if match else None

# UniProtID 추가
clin["UniProtID"] = clin["OtherIDs"].apply(extract_uniprot_id)

In [1]:
import pandas as pd

am_cols = [
    "CHROM", "POS", "REF", "ALT", "genome",
    "uniprot_id", "transcript_id", "protein_variant",
    "am_pathogenicity", "am_class"
]

am = pd.read_csv(
    r"E:\CAGI_data\AlphaMissense_hg38.tsv.gz",
    sep="\t",
    compression="infer",
    comment="#",
    names=am_cols,
    header=0  # 혹은 header=None + skiprows=1 로도 가능
)

In [39]:
# Step 1: ClinVar 컬럼 포맷 정리
clin["CHROM"] = "chr" + clin["Chromosome"].astype(str)
clin["POS"] = clin["Start"]
clin["REF"] = clin["ReferenceAlleleVCF"]
clin["ALT"] = clin["AlternateAlleleVCF"]
clin["genome"] = "hg38"

In [40]:
clin_key = clin[["CHROM", "POS", "REF", "ALT", "genome", "protein_variant", "Label"]]

In [42]:
clin_key = clin[["CHROM", "POS", "REF", "ALT", "genome", "protein_variant", "Label"]].copy()

# 중복 조합이 아예 없는 것만 추출
clin_counts = clin_key.groupby(["CHROM", "POS", "REF", "ALT","protein_variant"]).size()
clin_unique_keys = clin_counts[clin_counts == 1].reset_index()

In [43]:
clin_key

,CHROM,POS,REF,ALT,genome,protein_variant,Label
9,chr11,126277517,A,G,hg38,N430S,1
13,chr6,26092913,G,A,hg38,C282Y,1
31,chr6,26093215,G,T,hg38,R330M,1
33,chr6,26092916,A,C,hg38,Q283P,1
37,chr2,19945787,T,C,hg38,E615G,1
...,...,...,...,...,...,...,...
7449975,chr17,37739543,C,A,hg38,Q147H,1
7449979,chr17,37739491,G,C,hg38,R165G,1
7449981,chr17,37739620,C,A,hg38,A122S,1
7449985,chr13,94576332,C,T,hg38,E322K,1


In [44]:
clin_unique_keys

,CHROM,POS,REF,ALT,protein_variant,0
0,chr1,69134,A,G,E36G,1
1,chr1,925969,C,T,P189S,1
2,chr1,930165,G,A,R207Q,1
3,chr1,930204,G,A,R220Q,1
4,chr1,930245,G,A,D234N,1
...,...,...,...,...,...,...
196114,chrY,13323634,C,T,R1066H,1
196115,chrY,57087038,G,A,G122S,1
196116,chrY,57087054,T,C,M127T,1
196117,chrY,57196354,G,A,G331R,1


In [6]:
# am_key = am[["CHROM", "POS", "REF", "ALT", "protein_variant", "uniprot_id"]].copy()
am_counts = am.groupby(["CHROM", "POS", "REF", "ALT", "protein_variant"]).size()
am_unique_keys = am_counts[am_counts == 1].reset_index()

In [7]:
am

,CHROM,POS,REF,ALT,genome,uniprot_id,transcript_id,protein_variant,am_pathogenicity,am_class
0,chr1,69094,G,C,hg38,Q8NH21,ENST00000335137.4,V2L,0.2937,likely_benign
1,chr1,69094,G,A,hg38,Q8NH21,ENST00000335137.4,V2M,0.3296,likely_benign
2,chr1,69095,T,C,hg38,Q8NH21,ENST00000335137.4,V2A,0.2609,likely_benign
3,chr1,69095,T,A,hg38,Q8NH21,ENST00000335137.4,V2E,0.2922,likely_benign
4,chr1,69095,T,G,hg38,Q8NH21,ENST00000335137.4,V2G,0.2030,likely_benign
...,...,...,...,...,...,...,...,...,...,...
71697550,chrY,57196925,T,G,hg38,Q01113,ENST00000244174.10_PAR_Y,F521C,0.1903,likely_benign
71697551,chrY,57196925,T,C,hg38,Q01113,ENST00000244174.10_PAR_Y,F521S,0.2045,likely_benign
71697552,chrY,57196925,T,A,hg38,Q01113,ENST00000244174.10_PAR_Y,F521Y,0.1440,likely_benign
71697553,chrY,57196926,C,G,hg38,Q01113,ENST00000244174.10_PAR_Y,F521L,0.5879,likely_pathogenic


In [1]:
am_counts[am_counts == 1].reset_index()

NameError: name 'am_counts' is not defined

In [ ]:
import pandas as pd
import re

# 1. ClinVar 키
clin_key = clin[["Chromosome", "PositionVCF", "REF", "ALT", "protein_variant", "Label"]].copy()
clin_key["CHROM"] = "chr" + clin_key["Chromosome"].astype(str)
clin_key["POS"] = clin_key["PositionVCF"]
clin_key = clin_key[["CHROM", "POS", "REF", "ALT", "protein_variant", "Label"]]

# 중복 제거
clin_counts = clin_key.groupby(["CHROM", "POS", "REF", "ALT", "protein_variant"]).size()
clin_unique_keys = clin_counts[clin_counts == 1].reset_index()[["CHROM", "POS", "REF", "ALT","protein_variant"]]

# 2. AM 키
am_key = am[["CHROM", "POS", "REF", "ALT", "protein_variant", "uniprot_id"]].copy()
am_counts = am_key.groupby(["CHROM", "POS", "REF", "ALT", "protein_variant"]).size()
am_unique_keys = am_counts[am_counts == 1].reset_index()[["CHROM", "POS", "REF", "ALT", "protein_variant"]]


In [53]:
# 3. 교집합 키 기준 merge
key_merged = pd.merge(
    clin_unique_keys,
    am_unique_keys,
    on=["CHROM", "POS", "REF", "ALT", "protein_variant"],
    how="inner"
)

# 4. 정보 복원: Label, uniprot_id 가져오기
key_merged = pd.merge(key_merged, clin_key, on=["CHROM", "POS", "REF", "ALT", "protein_variant"], how="left")
key_merged = pd.merge(key_merged, am_key, on=["CHROM", "POS", "REF", "ALT", "protein_variant"], how="left")

In [54]:
def parse_variant(pv):
    match = re.match(r"^([A-Z])(\d+)([A-Z])$", pv)
    if match:
        return pd.Series({
            "WT": match.group(1),
            "MutPos": int(match.group(2)),
            "Mut": match.group(3)
        })
    else:
        return pd.Series({"WT": None, "MutPos": None, "Mut": None})

parsed = key_merged["protein_variant"].apply(parse_variant)
final_df = pd.concat([key_merged[["uniprot_id", "Label"]].reset_index(drop=True), parsed], axis=1)
final_df = final_df.dropna()
final_df = final_df.rename(columns={"uniprot_id": "UniProtID"})[["UniProtID", "MutPos", "WT", "Mut", "Label"]]

In [71]:
final_df = final_df.drop_duplicates()

In [ ]:
final_df.to_csv("filtered_variants_250822.tsv", sep="\t", index=False)

In [61]:
final_df_before = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants.tsv", sep="\t")

In [73]:
final_df_before["UniProtID"].unique().shape

(13599,)

In [3]:
import pandas as pd

final_df = pd.read_csv("filtered_variants_250822.tsv", sep="\t")

In [4]:
final_df["UniProtID"].unique().shape

(13853,)

In [6]:
final_df

,UniProtID,MutPos,WT,Mut,Label
0,Q9Y3T9,720,A,T,0
1,Q9Y3T9,708,E,D,0
2,Q9Y3T9,702,D,E,0
3,Q9Y3T9,693,R,W,0
4,Q9Y3T9,275,R,Q,0
...,...,...,...,...,...
172589,O00507,2077,V,I,0
172590,P51809,122,G,S,0
172591,P51809,127,M,T,0
172592,Q01113,331,G,R,0


In [7]:
import json
import os
import pandas as pd

# 경로 설정
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
# output_dir = r"E:\CAGI_data\fasta_files"
# os.makedirs(output_dir, exist_ok=True)

# JSON 불러오기
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# 필요한 ID 추출
unique_ids = final_df["UniProtID"].unique()

# 저장 및 매핑되지 않은 ID 추적
unmatched_ids = []

for uid in unique_ids:
    if uid in id_to_seq:
        pass
        # fasta_path = os.path.join(output_dir, f"{uid}.fasta")
        # with open(fasta_path, "w") as f:
        #     f.write(f">{uid}\n{id_to_seq[uid]}\n")
    else:
        unmatched_ids.append(uid)

# 매핑 안 된 ID 출력
if unmatched_ids:
    print(f"\n[❗] 다음 UniProt ID는 JSON에 매핑되지 않았습니다 ({len(unmatched_ids)}개):")
    for uid in unmatched_ids:
        print(uid)
else:
    print("\n✅ 모든 UniProt ID가 JSON에 매핑되었습니다.")


[❗] 다음 UniProt ID는 JSON에 매핑되지 않았습니다 (6개):
Q5VWM5
Q32Q52
A0A0B4J2F2
P0DN76
Q8N1N5
A0A1B0GWI6


In [8]:
import gzip
from Bio import SeqIO
import json

# 경로
trembl_path = r"E:\CAGI_data\uniprot_trembl.fasta.gz"
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
target_ids = {"A0A1B0GWI6"}

# 기존 JSON 로딩
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# FASTA에서 필요한 ID만 추출하여 추가
found = {}
with gzip.open(trembl_path, "rt") as handle:
    for record in SeqIO.parse(handle, "fasta"):
        uid = record.id.split("|")[1] if "|" in record.id else record.id.split()[0]
        if uid in target_ids:
            found[uid] = str(record.seq)
            print(f"✅ Found: {uid}")
        if len(found) == len(target_ids):
            break

# 결과 확인 및 병합
missing = target_ids - found.keys()

if missing:
    print(f"\n[❗] 다음 ID는 FASTA에서 찾지 못함: {missing} + 'A0A0B4J2F2', 'Q32Q52', 'Q8N1N5', 'Q5VWM5', 'P0DN76'")
else:
    print("\n✅ 모든 타겟 ID를 FASTA에서 찾음")

# JSON 병합 후 저장
id_to_seq.update(found)

with open(json_path, "w") as f:    
    json.dump(id_to_seq, f)

print("\n📦 JSON 파일이 성공적으로 업데이트되었습니다.")


KeyboardInterrupt: 

In [56]:
import pandas as pd

# 문제 ID 목록
invalid_ids = {'A0A0B4J2F2', 'Q32Q52', 'Q8N1N5', 'Q5VWM5', 'P0DN76'}

# ClinVar-AlphaMissense 기반 TSV 불러오기
filtered = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants.tsv", sep="\t")

# 해당 ID 제거
filtered = filtered[~filtered["UniProtID"].isin(invalid_ids)].reset_index(drop=True)
print(f"✅ After filtering, shape: {filtered.shape}")

filtered.to_csv("filtered_variants_cleaned.tsv", sep="\t", index=False)

✅ After filtering, shape: (156212, 5)


In [73]:
import pandas as pd
import json

# 경로 설정
filtered_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned.tsv"
reference_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\rhapsody2_sav_db_exactmatch_only.tsv"
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"

# 파일 로드
filtered = pd.read_csv(filtered_path, sep="\t")
df = pd.read_csv(reference_path, sep="\t", header=None)
df.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 유니프로트 ID → 시퀀스 매핑 로드
with open(json_path, "r") as f:
    uid_to_seq = json.load(f)

# FRAGMENTS 정의 (1–1400, 201–1600, ..., 5801–7200)
FRAGMENTS = {
    f"F{i+1}": (1 + i * 200, 1400 + i * 200) for i in range(100)
}

# fragment 선택 함수
def choose_structure_and_pdbpos(uid, mutpos):
    seq = uid_to_seq.get(uid, "")
    if not seq:
        return None, None  # 유니프로트 시퀀스 없음
    seq_len = len(seq)

    # 2700 이하면 F1으로 고정
    if seq_len <= 2700:
        return f"AF-{uid}-F1-model_v4.pdb", mutpos

    # 2701 이상이면 적절한 fragment 선택
    candidates = []
    for frag, (start, end) in FRAGMENTS.items():
        if start <= mutpos <= end:
            margin = min(mutpos - start, end - mutpos)
            candidates.append((frag, margin, start))
    if not candidates:
        return None, None
    # 가장 중앙에 가까운 fragment 선택
    chosen_frag, _, frag_start = max(candidates, key=lambda x: x[1])
    pdb_pos = mutpos - frag_start + 1
    structure_file = f"AF-{uid}-{chosen_frag}-model_v4.pdb"
    return structure_file, pdb_pos

# 결과 열 추가
structure_files = []
pdb_positions = []

for i, row in filtered.iterrows():
    uid, mutpos = row["UniProtID"], row["MutPos"]
    structure_file, pdb_pos = choose_structure_and_pdbpos(uid, mutpos)
    structure_files.append(structure_file)
    pdb_positions.append(pdb_pos)

filtered["StructureFile"] = structure_files
filtered["MutPos(pdb)"] = pdb_positions

filtered["MutPos(pdb)"] = filtered["MutPos(pdb)"].astype("Int64")

# 저장
output_path = filtered_path.replace(".tsv", "_with_structures.tsv")
filtered.to_csv(output_path, sep="\t", index=False)
print(f"✅ Saved to: {output_path}")

✅ Saved to: C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_with_structures.tsv


In [ ]:
df = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_with_structures.tsv", sep="\t")
df

,UniProtID,MutPos,WT,Mut,Label,StructureFile,MutPos(pdb)
0,Q9Y3T9,720,A,T,0,AF-Q9Y3T9-F1-model_v4.pdb,720
1,Q9Y3T9,693,R,W,0,AF-Q9Y3T9-F1-model_v4.pdb,693
2,Q9Y3T9,275,R,Q,0,AF-Q9Y3T9-F1-model_v4.pdb,275
3,Q9Y3T9,203,A,V,0,AF-Q9Y3T9-F1-model_v4.pdb,203
4,Q9Y3T9,194,N,S,0,AF-Q9Y3T9-F1-model_v4.pdb,194
...,...,...,...,...,...,...,...
156207,O00507,1060,A,T,0,AF-O00507-F1-model_v4.pdb,1060
156208,O00507,2077,V,I,0,AF-O00507-F1-model_v4.pdb,2077
156209,P51809,122,G,S,0,AF-P51809-F1-model_v4.pdb,122
156210,P51809,127,M,T,0,AF-P51809-F1-model_v4.pdb,127


In [83]:
df["StructureFile"].unique().shape

(15712,)

In [76]:
import pandas as pd
import tarfile
import os
import gzip
import shutil

# 경로 설정
tar_path = r"E:\CAGI_data\UP000005640_9606_HUMAN_v4.tar"
save_dir = r"E:\CAGI_data\pdb_files"
os.makedirs(save_dir, exist_ok=True)

# TSV 로드
df = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_with_structures.tsv", sep="\t")

# 필요한 StructureFile 목록 (중복 제거)
structure_files = set(df["StructureFile"].dropna().unique())

# 실패한 파일 목록
failed = []

# tar 열기
with tarfile.open(tar_path, "r") as tar:
    members = tar.getmembers()
    pdb_gz_names = {m.name: m for m in members if m.name.endswith(".pdb.gz")}

    for sfile in structure_files:
        pdb_gz_name = f"{sfile}.gz"  # ex: AF-Q9Y3T9-F1-model_v4.pdb.gz
        if pdb_gz_name in pdb_gz_names:
            member = pdb_gz_names[pdb_gz_name]
            try:
                # 압축 해제 대상 경로
                temp_gz_path = os.path.join(save_dir, pdb_gz_name)
                final_pdb_path = os.path.join(save_dir, sfile)

                # 1. .gz 파일 임시 추출
                with open(temp_gz_path, "wb") as out_f:
                    out_f.write(tar.extractfile(member).read())

                # 2. .gz → .pdb로 압축 해제
                with gzip.open(temp_gz_path, "rb") as f_in:
                    with open(final_pdb_path, "wb") as f_out:
                        shutil.copyfileobj(f_in, f_out)

                # 3. 임시 .gz 삭제
                os.remove(temp_gz_path)

            except Exception as e:
                print(f"❌ Failed: {sfile} — {e}")
                failed.append(sfile)
        else:
            print(f"❌ Missing in tar: {pdb_gz_name}")
            failed.append(sfile)

# 실패 로그 저장
if failed:
    log_path = os.path.join(save_dir, "failed_files.txt")
    with open(log_path, "w") as f:
        for name in failed:
            f.write(f"{name}\n")
    print(f"⚠ 실패한 파일 {len(failed)}개 → {log_path} 에 저장됨")

print("✅ 완료")


❌ Missing in tar: AF-P35556-F11-model_v4.pdb.gz
❌ Missing in tar: AF-Q9NRC6-F14-model_v4.pdb.gz
❌ Missing in tar: AF-Q9Y6V0-F22-model_v4.pdb.gz
❌ Missing in tar: AF-Q8TDX9-F10-model_v4.pdb.gz
❌ Missing in tar: AF-O60673-F13-model_v4.pdb.gz
❌ Missing in tar: AF-Q02388-F10-model_v4.pdb.gz
❌ Missing in tar: AF-Q92736-F22-model_v4.pdb.gz
❌ Missing in tar: AF-P01266-F9-model_v4.pdb.gz
❌ Missing in tar: AF-Q6ZRS2-F13-model_v4.pdb.gz
❌ Missing in tar: AF-Q96T58-F16-model_v4.pdb.gz
❌ Missing in tar: AF-Q9P2D7-F18-model_v4.pdb.gz
❌ Missing in tar: AF-P11532-F14-model_v4.pdb.gz
❌ Missing in tar: AF-O15018-F11-model_v4.pdb.gz
❌ Missing in tar: AF-Q5T011-F14-model_v4.pdb.gz
❌ Missing in tar: AF-Q07954-F20-model_v4.pdb.gz
❌ Missing in tar: AF-Q86WI1-F17-model_v4.pdb.gz
❌ Missing in tar: AF-Q63HN8-F23-model_v4.pdb.gz
❌ Missing in tar: AF-P25391-F12-model_v4.pdb.gz
❌ Missing in tar: AF-Q96PZ7-F13-model_v4.pdb.gz
❌ Missing in tar: AF-O60494-F15-model_v4.pdb.gz
❌ Missing in tar: AF-Q6ZS81-F12-model_v4.

총 샘플 수: 156212
FASTA 기준 일치 수: 156118 (99.94%)
PDB 기준 일치 수:   153832 (98.48%)
PDB 파싱 실패 수:    2309
불일치 총 2425건 → C:\Users\Kunny\Research\Project\BiConVarNet\failed_fasta_pdb_match.tsv에 저장

In [1]:
import pandas as pd

# 전체 TSV 로드
df = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_with_structures.tsv", sep="\t")

# 실패 로그 로드
fail_df = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\failed_fasta_pdb_match.tsv", sep="\t")

# 실패 샘플 제거
df_cleaned = df.merge(fail_df[["UniProtID", "MutPos", "WT", "StructureFile"]], 
                      on=["UniProtID", "MutPos", "WT", "StructureFile"], 
                      how="left", indicator=True)
df_cleaned = df_cleaned[df_cleaned["_merge"] == "left_only"].drop(columns=["_merge"])


In [6]:
df_cleaned.to_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_final.tsv", sep="\t", index=False)

In [7]:
df = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_final.tsv", sep="\t")

In [11]:
df

,UniProtID,MutPos,WT,Mut,Label,StructureFile,MutPos(pdb)
0,Q9Y3T9,720,A,T,0,AF-Q9Y3T9-F1-model_v4.pdb,720
1,Q9Y3T9,693,R,W,0,AF-Q9Y3T9-F1-model_v4.pdb,693
2,Q9Y3T9,275,R,Q,0,AF-Q9Y3T9-F1-model_v4.pdb,275
3,Q9Y3T9,203,A,V,0,AF-Q9Y3T9-F1-model_v4.pdb,203
4,Q9Y3T9,194,N,S,0,AF-Q9Y3T9-F1-model_v4.pdb,194
...,...,...,...,...,...,...,...
153782,O00507,1060,A,T,0,AF-O00507-F1-model_v4.pdb,1060
153783,O00507,2077,V,I,0,AF-O00507-F1-model_v4.pdb,2077
153784,P51809,122,G,S,0,AF-P51809-F1-model_v4.pdb,122
153785,P51809,127,M,T,0,AF-P51809-F1-model_v4.pdb,127


In [13]:
import json
import os
import pandas as pd

# 경로 설정
json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"
tsv_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_final.tsv"
output_dir = r"E:\CAGI_data\fasta_files_new"
os.makedirs(output_dir, exist_ok=True)

# JSON 불러오기
with open(json_path, "r") as f:
    id_to_seq = json.load(f)

# TSV 불러오기
df = pd.read_csv(tsv_path, sep="\t")
# 필요한 ID 추출
unique_ids = df["UniProtID"].unique()

# 저장 및 매핑되지 않은 ID 추적
unmatched_ids = []

for uid in unique_ids:
    if uid in id_to_seq:
        fasta_path = os.path.join(output_dir, f"{uid}.fasta")
        with open(fasta_path, "w") as f:
            f.write(f">{uid}\n{id_to_seq[uid]}\n")
    else:
        unmatched_ids.append(uid)

# 매핑 안 된 ID 출력
if unmatched_ids:
    print(f"\n[❗] 다음 UniProt ID는 JSON에 매핑되지 않았습니다 ({len(unmatched_ids)}개):")
    for uid in unmatched_ids:
        print(uid)
else:
    print("\n✅ 모든 UniProt ID가 JSON에 매핑되었습니다.")


✅ 모든 UniProt ID가 JSON에 매핑되었습니다.


In [6]:
am_cols = [
    "CHROM", "POS", "REF", "ALT", "genome",
    "uniprot_id", "transcript_id", "protein_variant",
    "am_pathogenicity", "am_class"
]

am = pd.read_csv(
    r"E:\CAGI_data\AlphaMissense_hg38.tsv.gz",
    sep="\t",
    compression="infer",
    comment="#",
    names=am_cols,
    header=0  # 혹은 header=None + skiprows=1 로도 가능
)

In [7]:
am["uniprot_id"].unique().shape

(19117,)

In [ ]:
am["uniprot_id"]

In [1]:
import pandas as pd

# 1. 파일 불러오기
filtered = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_final.tsv", sep="\t")

rhapsody = pd.read_csv(r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\rhapsody2_sav_db_exactmatch_only.tsv", sep="\t", header=None)
rhapsody.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 2. 공통 키 생성
filtered["key"] = (
    filtered["UniProtID"] + "_" +
    filtered["MutPos"].astype(str) + "_" +
    filtered["WT"] + "_" +
    filtered["Mut"]
)

rhapsody["key"] = (
    rhapsody["UniProtID"] + "_" +
    rhapsody["MutPos"].astype(str) + "_" +
    rhapsody["WT"] + "_" +
    rhapsody["Mut"]
)

# 3. 매칭 비율 계산 (rhapsody 기준)
matched_keys = set(filtered["key"]) & set(rhapsody["key"])
rhapsody["matched"] = rhapsody["key"].isin(matched_keys)

n_total = len(rhapsody)
n_matched = rhapsody["matched"].sum()
percent = n_matched / n_total * 100

print(f"Matched: {n_matched} / {n_total} ({percent:.2f}%)")


Matched: 81605 / 99855 (81.72%)


In [2]:
import pandas as pd

# 1. 파일 불러오기
filtered = pd.read_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\filtered_variants_cleaned_final.tsv", sep="\t")

rhapsody = pd.read_csv(r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\rhapsody2_sav_db_exactmatch_only.tsv", sep="\t", header=None)
rhapsody.columns = ["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label"]

# 2. 공통 키 생성
filtered["key"] = (
    filtered["UniProtID"] + "_" +
    filtered["MutPos"].astype(str) + "_" +
    filtered["WT"] + "_" +
    filtered["Mut"]
)

rhapsody["key"] = (
    rhapsody["UniProtID"] + "_" +
    rhapsody["MutPos"].astype(str) + "_" +
    rhapsody["WT"] + "_" +
    rhapsody["Mut"]
)

# 3. inner join으로 매칭된 데이터만 추출
merged = pd.merge(
    rhapsody[["key", "Label"]],
    filtered[["key", "Label"]],
    on="key",
    suffixes=("_rhap", "_filt")
)

# 4. Label 일치 여부 확인
merged["label_match"] = merged["Label_rhap"] == merged["Label_filt"]

n_total = len(merged)
n_matched = merged["label_match"].sum()
percent = n_matched / n_total * 100

print(f"Label matched: {n_matched} / {n_total} ({percent:.2f}%)")

# 혹시 불일치된 경우도 보고 싶다면:
mismatches = merged[~merged["label_match"]]
print("\nMismatched examples:")
print(mismatches.head())


Label matched: 81641 / 81648 (99.99%)

Mismatched examples:
                  key  Label_rhap  Label_filt  label_match
2484   O14647_730_K_R           1           0        False
2935   O14936_829_Q_R           1           0        False
11589  P01008_147_T_A           0           1        False
15467   P06280_80_G_A           1           0        False
23841  P22304_155_F_L           0           1        False


In [10]:
# 1. filtered와 rhapsody 모두 'key' 생성은 이미 되어 있으니 생략 가능 (안 했으면 아래 참고)
# filtered["key"] = ...
# rhapsody["key"] = ...

# 2. 매칭 비율 계산 (filtered 기준)
filtered["matched"] = filtered["key"].isin(set(rhapsody["key"]))

n_total_filtered = len(filtered)
n_matched_filtered = filtered["matched"].sum()
percent_filtered = n_matched_filtered / n_total_filtered * 100

print(f"Matched: {n_matched_filtered} / {n_total_filtered} ({percent_filtered:.2f}%)")


Matched: 81648 / 153787 (53.09%)
